In [14]:
from pathlib import Path
from datetime import timedelta
import polars as pl
import torch
from torch import nn

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

data = pl.read_csv(
    root / "data/raw/train.csv",
    try_parse_dates=True,
)

validation_start = data["date"].max() - timedelta(days=15) # type: ignore

train_data = data.filter(pl.col("date") < validation_start)
val_data = data.filter(pl.col("date") >= validation_start)

for name, frame in [("Training", train_data), ("Validation", val_data)]:
    print(
        name,
        frame.shape,
        frame["date"].min(),
        "to",
        frame["date"].max(),
    )

Training (2972376, 6) 2013-01-01 to 2017-07-30
Validation (28512, 6) 2017-07-31 to 2017-08-15


In [15]:
recent_start = validation_start - timedelta(days=28)
keys = ["store_nbr", "family", "weekday"]

baseline = (
    train_data
    .filter(pl.col("date") >= recent_start)
    .with_columns(pl.col("date").dt.weekday().alias("weekday"))
    .group_by(keys)
    .agg(pl.col("sales").log1p().mean().alias("pred_log_sales"))
)

scored = (
    val_data
    .with_columns(pl.col("date").dt.weekday().alias("weekday"))
    .join(baseline, on=keys, how="left", validate="m:1")
)

assert scored.height == val_data.height
assert scored["pred_log_sales"].null_count() == 0

baseline_rmsle = scored.select(
    (pl.col("sales").log1p() - pl.col("pred_log_sales"))
    .pow(2)
    .mean()
    .sqrt()
).item()

print(f"Same-weekday baseline RMSLE: {baseline_rmsle:.4f}")

Same-weekday baseline RMSLE: 0.5398


In [16]:
import numpy as np

keys = ["store_nbr", "family"]
series_keys = train_data.select(keys).unique().sort(keys)

first_date = train_data["date"].min()
calendar = pl.DataFrame({
    "date": pl.date_range(first_date, data["date"].max(), eager=True) # type: ignore
})

panel = (
    series_keys.join(calendar, how="cross")
    .join(
        data.select(keys + ["date", "sales", "onpromotion"]),
        on=keys + ["date"],
        how="left",
        validate="1:1",
    )
    .sort(keys + ["date"])
)

shape = (series_keys.height, calendar.height)
sales_observed = panel["sales"].is_not_null().to_numpy().reshape(shape)
log_sales = (
    panel["sales"].fill_null(0).log1p().cast(pl.Float32)
    .to_numpy().reshape(shape)
)
log_promotions = (
    panel["onpromotion"].fill_null(0).log1p().cast(pl.Float32)
    .to_numpy().reshape(shape)
)

lookback = 56
horizon = 16
split_day = (validation_start - first_date).days # type: ignore

history = slice(split_day - lookback, split_day)
forecast_days = slice(split_day, split_day + horizon)

print("Calendar panel:", log_sales.shape)
print("Validation sales history:", log_sales[:, history].shape)
print("Validation scheduled promotions:", log_promotions[:, forecast_days].shape)
print("Validation sales targets:", log_sales[:, forecast_days].shape)
print("All validation targets observed:", sales_observed[:, forecast_days].all())

Calendar panel: (1782, 1688)
Validation sales history: (1782, 56)
Validation scheduled promotions: (1782, 16)
Validation sales targets: (1782, 16)
All validation targets observed: True


In [17]:
series_count = log_sales.shape[0]

train_origins = np.arange(lookback, split_day - horizon + 1, 5)
history_offsets = np.arange(-lookback, 0)
target_offsets = np.arange(horizon)
example_count = len(train_origins) * series_count


def make_batch(origins, series_ids):
    rows = series_ids[:, None]
    past = origins[:, None] + history_offsets
    future = origins[:, None] + target_offsets

    return {
        "series_id": series_ids,
        "past_sales": log_sales[rows, past],
        "past_observed": sales_observed[rows, past],
        "future_promotions": log_promotions[rows, future],
        "target_sales": log_sales[rows, future],
        "target_observed": sales_observed[rows, future],
    }


def training_batch(example_ids):
    origin_indices, series_ids = np.divmod(example_ids, series_count)
    return make_batch(train_origins[origin_indices], series_ids)


last_target_day = int(train_origins[-1] + horizon - 1)
assert last_target_day < split_day

batch = training_batch(np.arange(256))
print("Training examples:", f"{example_count:,}")
print("Last training target date:", calendar["date"][last_target_day])
for name, values in batch.items():
    print(name, values.shape)

Training examples: 572,022
Last training target date: 2017-07-30
series_id (256,)
past_sales (256, 56)
past_observed (256, 56)
future_promotions (256, 16)
target_sales (256, 16)
target_observed (256, 16)


In [18]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_names = (
    "series_id",
    "past_sales",
    "past_observed",
    "future_promotions",
)


class SalesForecaster(nn.Module):
    def __init__(self, series_count, lookback, horizon):
        super().__init__()
        embedding_size = 16
        self.series_embedding = nn.Embedding(series_count, embedding_size)
        self.network = nn.Sequential(
            nn.Linear(2 * lookback + horizon + embedding_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, horizon),
        )

        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)
        self.register_buffer("weekday_index", torch.arange(horizon) % 7)

    def forward(self, series_id, past_sales, past_observed, future_promotions):
        recent_sales = past_sales[:, -28:].reshape(-1, 4, 7)
        recent_observed = past_observed[:, -28:].reshape(-1, 4, 7)
        weekday_mean = (
            (recent_sales * recent_observed).sum(dim=1)
            / recent_observed.sum(dim=1).clamp_min(1)
        )
        baseline = weekday_mean[:, self.weekday_index]

        features = torch.cat([
            past_sales,
            past_observed.float(),
            future_promotions,
            self.series_embedding(series_id),
        ], dim=1)
        return baseline + self.network(features)


def to_torch_batch(batch):
    return {
        name: torch.as_tensor(values, device=device)
        for name, values in batch.items()
    }


model = SalesForecaster(series_count, lookback, horizon).to(device)
validation_batch = to_torch_batch(
    make_batch(np.full(series_count, split_day), np.arange(series_count))
)

model.eval()
with torch.no_grad():
    predictions = model(*(validation_batch[name] for name in feature_names))
    mask = validation_batch["target_observed"]
    errors = predictions[mask] - validation_batch["target_sales"][mask]
    initial_rmsle = errors.square().mean().sqrt().item()

print("Forecast shape:", tuple(predictions.shape))
print(f"RMSLE before training: {initial_rmsle:.4f}")

Forecast shape: (1782, 16)
RMSLE before training: 0.5398


In [19]:
torch.manual_seed(42)
rng = np.random.default_rng(42)

model = SalesForecaster(series_count, lookback, horizon).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

epochs = 8
batch_size = 1024
best_rmsle = float("inf")
best_epoch = None
best_weights = None

for epoch in range(1, epochs + 1):
    model.train()
    order = rng.permutation(example_count)
    epoch_error = torch.zeros((), device=device)
    epoch_observed = torch.zeros((), device=device)

    for start in range(0, example_count, batch_size):
        ids = order[start:start + batch_size]
        batch = to_torch_batch(training_batch(ids))

        optimizer.zero_grad(set_to_none=True)
        predictions = model(*(batch[name] for name in feature_names))

        mask = batch["target_observed"]
        squared_error = (predictions - batch["target_sales"]).square()
        error_sum = (squared_error * mask).sum()
        observed_count = mask.sum()
        loss = error_sum / observed_count.clamp_min(1)

        loss.backward()
        optimizer.step()

        epoch_error += error_sum.detach()
        epoch_observed += observed_count

    train_rmse = (epoch_error / epoch_observed).sqrt().item()

    model.eval()
    with torch.no_grad():
        predictions = model(
            *(validation_batch[name] for name in feature_names)
        ).clamp_min(0)

        mask = validation_batch["target_observed"]
        errors = predictions[mask] - validation_batch["target_sales"][mask]
        val_rmsle = errors.square().mean().sqrt().item()

    if val_rmsle < best_rmsle:
        best_rmsle = val_rmsle
        best_epoch = epoch
        best_weights = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }

    print(
        f"Epoch {epoch:02d} | Train log RMSE {train_rmse:.4f} "
        f"| Validation RMSLE {val_rmsle:.4f}",
        flush=True,
    )

model.load_state_dict(best_weights)
print(f"\nBest epoch: {best_epoch}")
print(f"Baseline RMSLE: {baseline_rmsle:.4f}")
print(f"Model RMSLE:    {best_rmsle:.4f}")

Epoch 01 | Train log RMSE 0.6139 | Validation RMSLE 0.4695
Epoch 02 | Train log RMSE 0.5600 | Validation RMSLE 0.4613
Epoch 03 | Train log RMSE 0.5406 | Validation RMSLE 0.4581
Epoch 04 | Train log RMSE 0.5293 | Validation RMSLE 0.4579
Epoch 05 | Train log RMSE 0.5207 | Validation RMSLE 0.4402
Epoch 06 | Train log RMSE 0.5126 | Validation RMSLE 0.4422
Epoch 07 | Train log RMSE 0.5052 | Validation RMSLE 0.4397
Epoch 08 | Train log RMSE 0.5001 | Validation RMSLE 0.4360

Best epoch: 8
Baseline RMSLE: 0.5398
Model RMSLE:    0.4360


In [20]:
last_origin = log_sales.shape[1] - horizon
full_origins = np.arange(lookback, last_origin + 1, 5)

if full_origins[-1] != last_origin:
    full_origins = np.append(full_origins, last_origin)

full_example_count = len(full_origins) * series_count

torch.manual_seed(42)
rng = np.random.default_rng(42)
final_model = SalesForecaster(series_count, lookback, horizon).to(device)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=0.001)

for epoch in range(1, best_epoch + 1):
    final_model.train()
    order = rng.permutation(full_example_count)
    epoch_error = torch.zeros((), device=device)
    epoch_observed = torch.zeros((), device=device)

    for start in range(0, full_example_count, batch_size):
        ids = order[start:start + batch_size]
        origin_indices, series_ids = np.divmod(ids, series_count)
        batch = to_torch_batch(make_batch(full_origins[origin_indices], series_ids))

        optimizer.zero_grad(set_to_none=True)
        predictions = final_model(*(batch[name] for name in feature_names))
        mask = batch["target_observed"]
        error_sum = ((predictions - batch["target_sales"]).square() * mask).sum()
        observed_count = mask.sum()

        loss = error_sum / observed_count.clamp_min(1)
        loss.backward()
        optimizer.step()

        epoch_error += error_sum.detach()
        epoch_observed += observed_count

    train_rmse = (epoch_error / epoch_observed).sqrt().item()
    print(f"Refit epoch {epoch:02d}/{best_epoch} | Train log RMSE {train_rmse:.4f}",
          flush=True)

output_dir = root / "data" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state_dict": {
        name: value.detach().cpu()
        for name, value in final_model.state_dict().items()
    },
    "series_keys": series_keys.rows(),
    "lookback": lookback,
    "horizon": horizon,
    "epochs": best_epoch,
    "validation_rmsle": best_rmsle,
}, output_dir / "model.pt")

print("Last labeled day used:", calendar["date"][-1])
print("Saved:", output_dir / "model.pt")

Refit epoch 01/8 | Train log RMSE 0.6153
Refit epoch 02/8 | Train log RMSE 0.5640
Refit epoch 03/8 | Train log RMSE 0.5436
Refit epoch 04/8 | Train log RMSE 0.5311
Refit epoch 05/8 | Train log RMSE 0.5208
Refit epoch 06/8 | Train log RMSE 0.5131
Refit epoch 07/8 | Train log RMSE 0.5066
Refit epoch 08/8 | Train log RMSE 0.5013
Last labeled day used: 2017-08-15
Saved: d:\Coding\yumeko\data\output\model.pt


In [21]:
raw_dir = root / "data" / "raw"
series_columns = ["store_nbr", "family"]

test = (
    pl.read_csv(raw_dir / "test.csv", try_parse_dates=True)
    .sort(series_columns + ["date"])
)
sample = pl.read_csv(raw_dir / "sample_submission.csv")

test_start = calendar["date"][-1] + timedelta(days=1)
expected_dates = pl.date_range(
    test_start, test_start + timedelta(days=horizon - 1), eager=True
)

assert test["date"].unique().sort().to_list() == expected_dates.to_list()
assert test.select(series_columns).unique().sort(series_columns).equals(series_keys)
assert test.select(series_columns + ["date"]).unique().height == test.height
assert test.height == series_count * horizon

future_promotions = (
    test["onpromotion"].log1p().cast(pl.Float32)
    .to_numpy().reshape(series_count, horizon).copy()
)
test_batch = to_torch_batch({
    "series_id": np.arange(series_count),
    "past_sales": np.ascontiguousarray(log_sales[:, -lookback:]),
    "past_observed": np.ascontiguousarray(sales_observed[:, -lookback:]),
    "future_promotions": future_promotions,
})

final_model.eval()
with torch.inference_mode():
    log_predictions = final_model(*(test_batch[name] for name in feature_names))
    sales_predictions = torch.expm1(log_predictions.clamp_min(0)).cpu().numpy()

assert sales_predictions.shape == (series_count, horizon)
assert np.isfinite(sales_predictions).all()

forecasts = test.select("id").with_columns(
    pl.Series("sales", sales_predictions.ravel())
)
submission = sample.select("id").join(
    forecasts, on="id", how="left", validate="1:1", maintain_order="left"
)

assert submission.height == test.height
assert submission["sales"].null_count() == 0

submission_path = output_dir / "submission.csv"
submission.write_csv(submission_path)
print(f"Saved {submission.height:,} predictions to {submission_path}")
print(submission.head())

Saved 28,512 predictions to d:\Coding\yumeko\data\output\submission.csv
shape: (5, 2)
┌─────────┬─────────────┐
│ id      ┆ sales       │
│ ---     ┆ ---         │
│ i64     ┆ f32         │
╞═════════╪═════════════╡
│ 3000888 ┆ 3.521382    │
│ 3000889 ┆ 0.0         │
│ 3000890 ┆ 5.137973    │
│ 3000891 ┆ 2744.852051 │
│ 3000892 ┆ 0.065906    │
└─────────┴─────────────┘
